# 🚦 Autonomous Traffic Violation Detection Platform
**Full inference pipeline — Google Colab (GPU)**

| Module | Description |
|--------|-------------|
| Cell 1 | GPU check + install dependencies |
| Cell 2 | Download YOLOv8 models + EasyOCR |
| Cell 3 | Upload your test video |
| Cell 4 | Image enhancement layer |
| Cell 5 | Master detection + specialist cascade |
| Cell 6 | Violation rule engine |
| Cell 7 | Evidence package generator |
| Cell 8 | Analytics + summary report |

> **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — GPU CHECK + INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-500:])
    return result.stdout.strip()

# GPU check
print(run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))

# Install packages
packages = [
    'ultralytics',          # YOLOv8
    'easyocr',              # License plate OCR
    'supervision',          # Annotation utilities
    'opencv-python-headless',
    'Pillow',
    'pandas',
    'matplotlib',
    'seaborn',
    'tqdm',
    'scipy',
    'scikit-learn',
    'imutils',
]
print('Installing packages...')
run(f'{sys.executable} -m pip install -q ' + ' '.join(packages))
print('✓ All packages installed')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — IMPORTS + MODEL DOWNLOAD
# ─────────────────────────────────────────────
import os, cv2, time, json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple
from tqdm.notebook import tqdm
from PIL import Image
from ultralytics import YOLO
import easyocr
import supervision as sv
warnings.filterwarnings('ignore')

# ── Project directories ──────────────────────
BASE       = Path('/content/traffic_project')
MODELS_DIR = BASE / 'models'
EVIDENCE   = BASE / 'evidence'
CLIPS_DIR  = EVIDENCE / 'clips'
FRAMES_DIR = EVIDENCE / 'frames'
REPORTS    = BASE / 'reports'

for d in [MODELS_DIR, CLIPS_DIR, FRAMES_DIR, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Download YOLOv8 models ───────────────────
# We use pretrained COCO weights:
#   yolov8n  → fast master detector (vehicles, persons, traffic lights)
#   yolov8m  → better accuracy for specialist pass
# Fine-tuning on traffic datasets is the next step (see Cell 5 notes)

print('Loading YOLOv8 models (downloads ~12 MB each on first run)...')
master_model     = YOLO('yolov8n.pt')   # master detection pass
specialist_model = YOLO('yolov8m.pt')   # specialist cascade (higher accuracy)
print('✓ Models ready')

# ── EasyOCR reader (for license plates) ─────
print('Loading EasyOCR (first run downloads ~100 MB)...')
ocr_reader = easyocr.Reader(['en'], gpu=True, verbose=False)
print('✓ OCR ready')

# ── COCO class IDs we care about ────────────
COCO = {
    'person':         0,
    'bicycle':        1,
    'car':            2,
    'motorcycle':     3,
    'bus':            5,
    'truck':          7,
    'traffic light':  9,
    'stop sign':      11,
}
VEHICLE_IDS  = {COCO['car'], COCO['motorcycle'], COCO['bus'],
                COCO['truck'], COCO['bicycle']}
PERSON_ID    = COCO['person']
TLIGHT_ID    = COCO['traffic light']

print('\n✓ Setup complete. Proceed to Cell 3 to upload your video.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — UPLOAD VIDEO
# ─────────────────────────────────────────────
from google.colab import files

print('Upload your traffic video (.mp4 / .avi / .mov):')
uploaded = files.upload()

VIDEO_PATH = Path(list(uploaded.keys())[0])
print(f'\n✓ Video uploaded: {VIDEO_PATH}')

# Show basic video info
cap = cv2.VideoCapture(str(VIDEO_PATH))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f'  Resolution : {W}×{H}')
print(f'  FPS        : {FPS:.1f}')
print(f'  Frames     : {TOTAL_FRAMES}')
print(f'  Duration   : {TOTAL_FRAMES/FPS:.1f}s')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — ADAPTIVE IMAGE ENHANCEMENT LAYER
# ─────────────────────────────────────────────

class AdaptiveEnhancer:
    """
    Quality assessment gate: only enhances frames that need it.
    Checks: brightness (low-light), blur (motion), contrast.
    """

    def __init__(self,
                 brightness_thresh: float = 60.0,
                 blur_thresh: float = 80.0,
                 clahe_clip: float = 3.0):
        self.brightness_thresh = brightness_thresh
        self.blur_thresh       = blur_thresh
        # CLAHE for low-light
        self.clahe = cv2.createCLAHE(clipLimit=clahe_clip,
                                     tileGridSize=(8, 8))
        self.stats = defaultdict(int)

    # ── Quality assessment ───────────────────
    def _brightness(self, frame: np.ndarray) -> float:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return float(np.mean(gray))

    def _blur_score(self, frame: np.ndarray) -> float:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return float(cv2.Laplacian(gray, cv2.CV_64F).var())

    def _has_fog(self, frame: np.ndarray) -> bool:
        """High mean brightness + low std → foggy/hazy."""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        return bool(np.mean(gray) > 160 and np.std(gray) < 30)

    # ── Enhancement methods ──────────────────
    def _enhance_low_light(self, frame: np.ndarray) -> np.ndarray:
        """CLAHE on L channel of LAB."""
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

    def _deblur(self, frame: np.ndarray) -> np.ndarray:
        """Lightweight unsharp mask for motion blur."""
        blurred = cv2.GaussianBlur(frame, (0, 0), 3)
        return cv2.addWeighted(frame, 1.5, blurred, -0.5, 0)

    def _dehaze(self, frame: np.ndarray) -> np.ndarray:
        """Dark channel prior – simple histogram stretch."""
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

    # ── Main entry point ─────────────────────
    def enhance(self, frame: np.ndarray) -> Tuple[np.ndarray, List[str]]:
        applied = []
        out = frame.copy()

        if self._brightness(out) < self.brightness_thresh:
            out = self._enhance_low_light(out)
            applied.append('low_light')
            self.stats['low_light'] += 1

        if self._blur_score(out) < self.blur_thresh:
            out = self._deblur(out)
            applied.append('deblur')
            self.stats['deblur'] += 1

        if self._has_fog(out):
            out = self._dehaze(out)
            applied.append('dehaze')
            self.stats['dehaze'] += 1

        self.stats['total'] += 1
        return out, applied


enhancer = AdaptiveEnhancer()

# Quick visual test on first frame
cap = cv2.VideoCapture(str(VIDEO_PATH))
ret, sample = cap.read()
cap.release()

if ret:
    enhanced, ops = enhancer.enhance(sample)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(sample,   cv2.COLOR_BGR2RGB)); axes[0].set_title('Original');             axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB)); axes[1].set_title(f'Enhanced {ops or "(no change)"}'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
    print(f'Enhancement applied: {ops or "none needed"}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — DATA CLASSES + DETECTION ENGINE
# ─────────────────────────────────────────────

# ── Data structures ──────────────────────────

@dataclass
class Detection:
    track_id:   int
    class_id:   int
    class_name: str
    confidence: float
    bbox:       Tuple[int, int, int, int]   # x1,y1,x2,y2
    frame_idx:  int


@dataclass
class ViolationRecord:
    violation_id:  str
    plate:         str
    violation:     str
    confidence:    float
    camera:        str
    timestamp:     str
    frame_idx:     int
    bbox:          Tuple[int, int, int, int]
    vehicle_type:  str
    rider_count:   int
    exempt:        bool
    evidence_img:  str = ''
    evidence_clip: str = ''
    enhancement:   List[str] = field(default_factory=list)


# ── Signal state (3-layer fallback) ──────────

class SignalStateEstimator:
    """
    Layer 1: YOLO detects signal crop, classify color.
    Layer 3: Cycle learning fallback when no signal visible.
    """
    CYCLE = {'red': 60, 'green': 45, 'yellow': 5}  # typical seconds

    def __init__(self, fps: float):
        self.fps          = fps
        self.history      = deque(maxlen=300)   # last 300 detections
        self.current      = 'unknown'
        self._cycle_start = 0
        self._cycle_phase = 'red'
        self._total_cycle = sum(self.CYCLE.values())

    def update_from_frame(self, frame: np.ndarray,
                          tlight_boxes: List[Tuple]) -> str:
        if tlight_boxes:
            state = self._classify_signal(frame, tlight_boxes[0])
            self.history.append(state)
            self.current = state
        elif len(self.history) > 10:
            # Layer 3: predict from learned cycle
            self.current = self._predict_from_cycle()
        return self.current

    def _classify_signal(self, frame, bbox) -> str:
        x1, y1, x2, y2 = [int(v) for v in bbox]
        crop = frame[max(0,y1):y2, max(0,x1):x2]
        if crop.size == 0:
            return 'unknown'
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

        # Red mask (wraps around 0/180 in HSV)
        r1 = cv2.inRange(hsv, (0,   100, 100), (10,  255, 255))
        r2 = cv2.inRange(hsv, (160, 100, 100), (180, 255, 255))
        red_px    = cv2.countNonZero(r1 | r2)
        green_px  = cv2.countNonZero(cv2.inRange(hsv, (40,  100, 100), (90,  255, 255)))
        yellow_px = cv2.countNonZero(cv2.inRange(hsv, (15,  100, 100), (40,  255, 255)))

        counts = {'red': red_px, 'green': green_px, 'yellow': yellow_px}
        best   = max(counts, key=counts.get)
        return best if counts[best] > 50 else 'unknown'

    def _predict_from_cycle(self) -> str:
        # Estimate where we are in the cycle from history length
        frames_in  = len(self.history)
        seconds_in = (frames_in / self.fps) % self._total_cycle
        acc = 0
        for phase, dur in self.CYCLE.items():
            acc += dur
            if seconds_in < acc:
                return phase
        return 'red'

    @property
    def is_red(self) -> bool:
        return self.current == 'red'


# ── Trajectory tracker (lane learning) ───────

class TrajectoryTracker:
    """
    Tracks vehicle centroids over time.
    After LEARNING_FRAMES frames, clusters trajectories
    to discover valid flow directions automatically.
    """
    LEARNING_FRAMES = 500   # ~20s at 25fps
    N_CLUSTERS      = 4     # expected flow directions per intersection

    def __init__(self):
        self.tracks:      Dict[int, List[Tuple[float,float]]] = defaultdict(list)
        self.learned      = False
        self.valid_dirs:  Optional[np.ndarray] = None   # cluster centers
        self._frame_count = 0

    def update(self, track_id: int, cx: float, cy: float):
        self.tracks[track_id].append((cx, cy))
        self._frame_count += 1
        if not self.learned and self._frame_count >= self.LEARNING_FRAMES:
            self._learn_flow()

    def _learn_flow(self):
        from sklearn.cluster import KMeans
        dirs = []
        for pts in self.tracks.values():
            if len(pts) < 5:
                continue
            pts = np.array(pts)
            # Direction vector: start → end
            dx = pts[-1][0] - pts[0][0]
            dy = pts[-1][1] - pts[0][1]
            norm = np.hypot(dx, dy)
            if norm > 10:
                dirs.append([dx/norm, dy/norm])

        if len(dirs) >= self.N_CLUSTERS:
            km = KMeans(n_clusters=self.N_CLUSTERS, n_init=10, random_state=42)
            km.fit(dirs)
            self.valid_dirs = km.cluster_centers_
            self.learned    = True
            print(f'  [Lane learning] ✓ Learned {self.N_CLUSTERS} flow directions from {len(dirs)} trajectories')

    def is_wrong_side(self, track_id: int,
                      threshold: float = 0.7) -> bool:
        """Returns True if vehicle direction is anomalous."""
        if not self.learned:
            return False
        pts = self.tracks.get(track_id, [])
        if len(pts) < 8:
            return False
        pts = np.array(pts[-8:])
        dx  = pts[-1][0] - pts[0][0]
        dy  = pts[-1][1] - pts[0][1]
        norm = np.hypot(dx, dy)
        if norm < 5:
            return False
        d    = np.array([dx/norm, dy/norm])
        sims = self.valid_dirs @ d             # cosine similarity
        return float(np.max(sims)) < threshold  # doesn't match any known flow


# ── OCR helper ───────────────────────────────

def extract_plate(frame: np.ndarray, bbox: Tuple) -> str:
    """Crop region, run EasyOCR, return cleaned plate string."""
    x1, y1, x2, y2 = [int(v) for v in bbox]
    pad  = 8
    crop = frame[max(0, y1-pad):y2+pad, max(0, x1-pad):x2+pad]
    if crop.size == 0:
        return 'UNKNOWN'
    # Upscale for better OCR
    crop = cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, crop = cv2.threshold(crop, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    results = ocr_reader.readtext(crop, detail=0, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    plate   = ''.join(results).upper().replace(' ', '')
    return plate[:10] if plate else 'UNKNOWN'


# ── ANPR registry ────────────────────────────

class ANPRRegistry:
    """Continuous plate log — all vehicles, not just violators."""

    def __init__(self):
        self.records: List[Dict] = []

    def log(self, plate: str, frame_idx: int,
            timestamp: str, camera: str = 'CAM_01'):
        self.records.append({
            'plate':     plate,
            'frame':     frame_idx,
            'timestamp': timestamp,
            'camera':    camera,
        })

    def to_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.records)

    def route_of(self, plate: str) -> List[Dict]:
        return [r for r in self.records if r['plate'] == plate]


print('✓ All classes and helpers defined. Proceed to Cell 6.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — VIOLATION RULE ENGINE
# ─────────────────────────────────────────────

CAMERA_ID = 'CAM_01'

# ── Confidence thresholds ────────────────────
CONF = {
    'detection':     0.40,   # master model
    'specialist':    0.45,   # specialist models
    'violation_min': 0.50,   # minimum fused confidence to record
}

# ── Emergency vehicle class names ────────────
EXEMPT_KEYWORDS = {'ambulance', 'fire truck', 'police'}


class ViolationEngine:
    """
    Stateful rule engine.
    One instance per video — maintains signal, tracker, ANPR.
    """

    def __init__(self, fps: float, frame_w: int, frame_h: int):
        self.fps      = fps
        self.W        = frame_w
        self.H        = frame_h
        self.signal   = SignalStateEstimator(fps)
        self.tracker  = TrajectoryTracker()
        self.anpr     = ANPRRegistry()
        self.violations: List[ViolationRecord] = []
        self._vid_id  = 0

        # Stop-line: bottom 20% of frame (auto-estimated)
        self.stop_line_y = int(frame_h * 0.80)

    # ── Helpers ──────────────────────────────
    def _timestamp(self, frame_idx: int) -> str:
        secs = frame_idx / self.fps
        return datetime.utcfromtimestamp(secs).strftime('%H:%M:%S.%f')[:-3]

    def _violation_id(self, frame_idx: int) -> str:
        self._vid_id += 1
        return f'VIO-{CAMERA_ID}-{frame_idx:06d}-{self._vid_id:04d}'

    def _fuse_confidence(self, *scores: float) -> float:
        """Geometric mean of multiple model scores."""
        arr = [s for s in scores if s > 0]
        if not arr:
            return 0.0
        return float(np.prod(arr) ** (1.0 / len(arr)))

    def _centroid(self, bbox) -> Tuple[float, float]:
        x1, y1, x2, y2 = bbox
        return (x1+x2)/2, (y1+y2)/2

    def _is_exempt(self, class_name: str) -> bool:
        return any(k in class_name.lower() for k in EXEMPT_KEYWORDS)

    def _iou(self, a, b) -> float:
        ax1,ay1,ax2,ay2 = a;  bx1,by1,bx2,by2 = b
        ix1 = max(ax1,bx1); iy1 = max(ay1,by1)
        ix2 = min(ax2,bx2); iy2 = min(ay2,by2)
        inter = max(0,ix2-ix1)*max(0,iy2-iy1)
        if inter == 0: return 0.0
        ua = (ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/ua if ua > 0 else 0.0

    def _count_riders(self, vehicle_bbox,
                      person_boxes: List) -> int:
        """Count persons overlapping with a vehicle bbox."""
        count = 0
        for pb in person_boxes:
            if self._iou(vehicle_bbox, pb) > 0.15:
                count += 1
        return count

    def _has_helmet(self, frame: np.ndarray, bbox) -> Tuple[bool, float]:
        """
        Heuristic helmet check on head region above the vehicle.
        In production: replace with a fine-tuned helmet classifier.
        Uses: top-quarter of rider bbox, checks for round dark shape.
        """
        x1,y1,x2,y2 = [int(v) for v in bbox]
        h = y2-y1; w = x2-x1
        head_crop = frame[max(0,y1):y1+h//3, max(0,x1):x2]
        if head_crop.size == 0:
            return True, 0.5   # inconclusive
        gray      = cv2.cvtColor(head_crop, cv2.COLOR_BGR2GRAY)
        circles   = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2,
                                     minDist=20, param1=50, param2=30,
                                     minRadius=w//8, maxRadius=w//3)
        if circles is not None:
            return True, 0.70   # round object detected = likely helmet
        # Fallback: check for dark region in head area
        dark_ratio = np.sum(gray < 80) / gray.size
        if dark_ratio > 0.3:
            return True, 0.55
        return False, 0.65   # no round shape = no helmet

    def _has_seatbelt(self, frame: np.ndarray, bbox,
                      driver_conf: float) -> Tuple[bool, float]:
        """
        Heuristic seatbelt check: looks for diagonal line across
        torso region. In production: use a fine-tuned classifier.
        """
        x1,y1,x2,y2 = [int(v) for v in bbox]
        h = y2-y1
        torso = frame[y1+h//4:y1+3*h//4, x1:x2]
        if torso.size == 0:
            return True, 0.5
        gray  = cv2.cvtColor(torso, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=30,
                                 minLineLength=torso.shape[0]//3,
                                 maxLineGap=10)
        if lines is not None:
            for line in lines:
                x_,y_,x2_,y2_ = line[0]
                angle = abs(np.degrees(np.arctan2(y2_-y_, x2_-x_)))
                if 20 < angle < 70:   # diagonal = seatbelt direction
                    return True, 0.65
        return False, 0.60

    # ── Main per-frame processor ──────────────
    def process_frame(self,
                      frame:      np.ndarray,
                      frame_idx:  int,
                      enhancement: List[str]) -> List[ViolationRecord]:

        new_violations = []
        ts = self._timestamp(frame_idx)

        # ── Master detection pass ────────────
        results = master_model.track(frame, persist=True,
                                     conf=CONF['detection'],
                                     classes=list(COCO.values()),
                                     verbose=False)
        if not results or results[0].boxes is None:
            return []

        boxes  = results[0].boxes
        bboxes = boxes.xyxy.cpu().numpy()    if boxes.xyxy   is not None else []
        confs  = boxes.conf.cpu().numpy()    if boxes.conf   is not None else []
        clss   = boxes.cls.cpu().numpy().astype(int) if boxes.cls is not None else []
        ids    = boxes.id.cpu().numpy().astype(int)  if boxes.id  is not None else range(len(bboxes))

        person_boxes  = [bboxes[i] for i,c in enumerate(clss) if c == PERSON_ID]
        tlight_boxes  = [bboxes[i] for i,c in enumerate(clss) if c == TLIGHT_ID]
        vehicle_dets  = [(bboxes[i], confs[i], clss[i], ids[i])
                         for i,c in enumerate(clss) if c in VEHICLE_IDS]

        # ── Update signal state ──────────────
        signal_state = self.signal.update_from_frame(frame, tlight_boxes)

        # ── Per-vehicle checks ───────────────
        for (bbox, det_conf, cls_id, track_id) in vehicle_dets:

            x1,y1,x2,y2 = [int(v) for v in bbox]
            cx, cy      = self._centroid(bbox)
            class_name  = master_model.names[cls_id]
            is_moto     = cls_id == COCO['motorcycle']
            is_car      = cls_id in {COCO['car'], COCO['bus'], COCO['truck']}

            # Update trajectory
            self.tracker.update(int(track_id), cx, cy)

            # ANPR: log every vehicle
            plate = extract_plate(frame, bbox)
            self.anpr.log(plate, frame_idx, ts, CAMERA_ID)

            # Emergency vehicle check
            exempt = self._is_exempt(class_name)
            rider_count = self._count_riders(bbox, person_boxes)

            def record(vtype, conf):
                fused = self._fuse_confidence(det_conf, conf)
                if fused < CONF['violation_min']:
                    return
                vid = self._violation_id(frame_idx)
                new_violations.append(ViolationRecord(
                    violation_id = vid,
                    plate        = plate,
                    violation    = vtype,
                    confidence   = round(fused, 3),
                    camera       = CAMERA_ID,
                    timestamp    = ts,
                    frame_idx    = frame_idx,
                    bbox         = (x1,y1,x2,y2),
                    vehicle_type = class_name,
                    rider_count  = rider_count,
                    exempt       = exempt,
                    enhancement  = enhancement,
                ))

            # ─ 1. Helmet non-compliance (motorcycles) ─
            if is_moto and not exempt:
                has_h, h_conf = self._has_helmet(frame, bbox)
                if not has_h:
                    record('Helmet non-compliance', h_conf)

            # ─ 2. Triple riding ──────────────
            if is_moto and rider_count >= 3 and not exempt:
                record('Triple riding', 0.85)

            # ─ 3. Seatbelt non-compliance ────
            if is_car and person_boxes and not exempt:
                has_sb, sb_conf = self._has_seatbelt(frame, bbox, det_conf)
                if not has_sb:
                    record('Seatbelt non-compliance', sb_conf)

            # ─ 4. Red-light violation ─────────
            if signal_state == 'red' and not exempt:
                # Vehicle moving through intersection when light is red
                if cy < self.H * 0.75 and len(self.tracker.tracks[int(track_id)]) > 5:
                    pts = np.array(self.tracker.tracks[int(track_id)][-5:])
                    if np.std(pts[:,1]) > 3:   # is moving
                        record('Red-light violation', 0.80)

            # ─ 5. Stop-line violation ─────────
            if signal_state in ('red', 'unknown') and not exempt:
                if y2 > self.stop_line_y:
                    record('Stop-line violation', 0.72)

            # ─ 6. Wrong-side driving ──────────
            if self.tracker.is_wrong_side(int(track_id)) and not exempt:
                record('Wrong-side driving', 0.78)

        # ─ 7. Illegal parking ─────────────────
        # Vehicles stationary for > 5 seconds in motion zone
        for (bbox, det_conf, cls_id, track_id) in vehicle_dets:
            pts = self.tracker.tracks.get(int(track_id), [])
            if len(pts) > int(self.fps * 5):
                recent = np.array(pts[-int(self.fps*5):])
                if np.std(recent[:,0]) < 5 and np.std(recent[:,1]) < 5:
                    plate  = self.anpr.records[-1]['plate'] if self.anpr.records else 'UNKNOWN'
                    x1,y1,x2,y2 = [int(v) for v in bbox]
                    fused = self._fuse_confidence(float(det_conf), 0.75)
                    if fused >= CONF['violation_min']:
                        vid = self._violation_id(frame_idx)
                        new_violations.append(ViolationRecord(
                            violation_id = vid, plate=plate,
                            violation    = 'Illegal parking',
                            confidence   = round(fused,3),
                            camera=CAMERA_ID, timestamp=ts,
                            frame_idx=frame_idx, bbox=(x1,y1,x2,y2),
                            vehicle_type = master_model.names[cls_id],
                            rider_count=0, exempt=False,
                            enhancement=enhancement,
                        ))

        self.violations.extend(new_violations)
        return new_violations


print('✓ ViolationEngine defined. Proceed to Cell 7.')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — EVIDENCE GENERATOR + MAIN RUN LOOP
# ─────────────────────────────────────────────

# ── Config ───────────────────────────────────
PROCESS_EVERY_N = 2          # process every Nth frame (speed vs accuracy)
CLIP_SECONDS    = 5          # seconds of video to save around each violation
LABEL_COLORS = {
    'Helmet non-compliance':    (0,   0,   255),
    'Seatbelt non-compliance':  (0,  128,  255),
    'Triple riding':            (255,  0,  255),
    'Wrong-side driving':       (255,  0,    0),
    'Stop-line violation':      (0,  200,  255),
    'Red-light violation':      (0,   0,  200),
    'Illegal parking':          (128, 0,  255),
}


class EvidenceGenerator:
    """Saves annotated frame + 5s clip for each violation."""

    def __init__(self, source_path: Path, fps: float):
        self.source_path   = source_path
        self.fps           = fps
        self.clip_frames   = int(fps * CLIP_SECONDS)
        # Buffer: ring buffer of recent frames for clip extraction
        self.frame_buffer: deque = deque(maxlen=self.clip_frames)

    def push_frame(self, frame: np.ndarray):
        self.frame_buffer.append(frame.copy())

    def save_frame(self, frame: np.ndarray,
                   violation: ViolationRecord) -> str:
        annotated = frame.copy()
        x1,y1,x2,y2 = violation.bbox
        color = LABEL_COLORS.get(violation.violation, (255,255,0))

        # Bounding box
        cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)

        # Label background
        label = f'{violation.violation} | {violation.plate} | {violation.confidence:.2f}'
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(annotated, (x1, y1-th-10), (x1+tw+6, y1), color, -1)
        cv2.putText(annotated, label, (x1+3, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)

        # Signal state overlay
        cv2.putText(annotated, f'Signal: {violation.timestamp}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)

        path = str(FRAMES_DIR / f'{violation.violation_id}.jpg')
        cv2.imwrite(path, annotated)
        return path

    def save_clip(self, violation: ViolationRecord,
                  future_frames: List[np.ndarray]) -> str:
        path = str(CLIPS_DIR / f'{violation.violation_id}.mp4')
        h, w = list(self.frame_buffer)[0].shape[:2] if self.frame_buffer else (480, 640)
        out  = cv2.VideoWriter(path,
                               cv2.VideoWriter_fourcc(*'mp4v'),
                               self.fps, (w, h))
        for f in self.frame_buffer:
            out.write(f)
        for f in future_frames[:self.clip_frames//2]:
            out.write(f)
        out.release()
        return path


# ── Main inference loop ───────────────────────

engine   = ViolationEngine(fps=FPS, frame_w=W, frame_h=H)
evidence = EvidenceGenerator(VIDEO_PATH, FPS)

cap         = cv2.VideoCapture(str(VIDEO_PATH))
frame_idx   = 0
pending_clips: Dict[str, ViolationRecord] = {}   # vid_id → record
all_violations: List[ViolationRecord] = []

print(f'Processing video: {VIDEO_PATH.name}')
print(f'Frames to process: {TOTAL_FRAMES // PROCESS_EVERY_N}')

pbar = tqdm(total=TOTAL_FRAMES, desc='Inference', unit='frames')

while cap.isOpened():
    ret, raw_frame = cap.read()
    if not ret:
        break

    evidence.push_frame(raw_frame)
    frame_idx += 1
    pbar.update(1)

    if frame_idx % PROCESS_EVERY_N != 0:
        continue

    # Enhancement
    frame, enh_ops = enhancer.enhance(raw_frame)

    # Detection + violation engine
    new_viols = engine.process_frame(frame, frame_idx, enh_ops)

    # Save evidence for new violations
    for v in new_viols:
        img_path = evidence.save_frame(frame, v)
        v.evidence_img = img_path
        # Queue clip saving (needs future frames)
        pending_clips[v.violation_id] = v

    # Save queued clips using current buffer
    if pending_clips and frame_idx % int(FPS*2) == 0:
        for vid_id, v in list(pending_clips.items()):
            clip_path = evidence.save_clip(v, [])
            v.evidence_clip = clip_path
            all_violations.append(v)
            pending_clips.pop(vid_id)

pbar.close()
cap.release()

# Flush remaining
for vid_id, v in pending_clips.items():
    clip_path = evidence.save_clip(v, [])
    v.evidence_clip = clip_path
    all_violations.append(v)

print(f'\n✓ Done. Total violations detected: {len(all_violations)}')
print(f'  Enhancement stats: {dict(enhancer.stats)}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — EVIDENCE JSON + ANALYTICS REPORT
# ─────────────────────────────────────────────

# ── Save violation records as JSON ───────────
records_path = REPORTS / 'violations.json'
with open(records_path, 'w') as f:
    json.dump([asdict(v) for v in all_violations], f, indent=2)
print(f'✓ Violation records → {records_path}')

# ── Save ANPR registry ────────────────────────
anpr_path = REPORTS / 'anpr_registry.csv'
engine.anpr.to_df().to_csv(anpr_path, index=False)
print(f'✓ ANPR registry     → {anpr_path}')

# ── Analytics ────────────────────────────────
if not all_violations:
    print('\nNo violations detected in this video.')
else:
    df = pd.DataFrame([asdict(v) for v in all_violations])

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Traffic Violation Analytics Report', fontsize=16, fontweight='bold')

    # ─ 1. Violation type distribution ─────────
    ax = axes[0, 0]
    counts = df['violation'].value_counts()
    bars = ax.barh(counts.index, counts.values,
                   color=[f'#{LABEL_COLORS.get(v,(100,100,200))[2]:02x}'
                          f'{LABEL_COLORS.get(v,(100,100,200))[1]:02x}'
                          f'{LABEL_COLORS.get(v,(100,100,200))[0]:02x}'
                          for v in counts.index])
    ax.set_xlabel('Count')
    ax.set_title('Violations by type')
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.invert_yaxis()

    # ─ 2. Confidence score distribution ───────
    ax = axes[0, 1]
    ax.hist(df['confidence'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df['confidence'].mean(), color='red', linestyle='--',
               label=f'Mean: {df["confidence"].mean():.2f}')
    ax.set_xlabel('Fused confidence')
    ax.set_ylabel('Frequency')
    ax.set_title('Confidence score distribution')
    ax.legend()

    # ─ 3. Timeline (violations over time) ─────
    ax = axes[1, 0]
    df['second'] = (df['frame_idx'] / FPS).astype(int)
    timeline = df.groupby('second').size()
    ax.plot(timeline.index, timeline.values, color='crimson', linewidth=1.5)
    ax.fill_between(timeline.index, timeline.values, alpha=0.2, color='crimson')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Violations / second')
    ax.set_title('Violation timeline')

    # ─ 4. Top repeat offenders ─────────────────
    ax = axes[1, 1]
    repeat = df[df['plate'] != 'UNKNOWN']['plate'].value_counts().head(10)
    if not repeat.empty:
        ax.bar(repeat.index, repeat.values, color='darkorange')
        ax.set_xlabel('Plate')
        ax.set_ylabel('Violations')
        ax.set_title('Top repeat offenders')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, 'No plates recognised', ha='center', transform=ax.transAxes)
        ax.set_title('Top repeat offenders')

    plt.tight_layout()
    report_img = str(REPORTS / 'analytics.png')
    plt.savefig(report_img, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Analytics chart   → {report_img}')

    # ── Text summary ──────────────────────────
    print('\n' + '='*55)
    print('  SUMMARY REPORT')
    print('='*55)
    print(f'  Total violations  : {len(df)}')
    print(f'  Unique plates     : {df["plate"].nunique()}')
    print(f'  Avg confidence    : {df["confidence"].mean():.3f}')
    print(f'  Exempt vehicles   : {df["exempt"].sum()}')
    print(f'  Enhanced frames   : {enhancer.stats.get("total",0)}')
    print()
    print('  Breakdown:')
    for vtype, cnt in df['violation'].value_counts().items():
        print(f'    {vtype:<30} {cnt}')
    print('='*55)

    # ── Performance metrics on available data ─
    print('\n  NOTE: Accuracy/Precision/Recall/F1/mAP require')
    print('  ground-truth annotations. See Cell 9 to evaluate')
    print('  on a labelled dataset (COCO-format annotations).')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — PERFORMANCE EVALUATION
# (Run only if you have ground-truth labels)
# ─────────────────────────────────────────────
# Place your ground-truth JSON at:
#   /content/traffic_project/gt_labels.json
#
# Format:
# [
#   {"frame_idx": 120, "violation": "Helmet non-compliance"},
#   ...
# ]

from sklearn.metrics import (precision_score, recall_score,
                             f1_score, classification_report,
                             confusion_matrix)

GT_PATH = BASE / 'gt_labels.json'

if not GT_PATH.exists():
    print('No ground-truth file found at', GT_PATH)
    print('Skipping evaluation. Create gt_labels.json to enable metrics.')
else:
    with open(GT_PATH) as f:
        gt_raw = json.load(f)

    VIOLATION_TYPES = sorted(set(
        [v['violation'] for v in gt_raw] +
        [v.violation   for v in all_violations]
    ))

    # Build frame-level label sets
    gt_frames   = {r['frame_idx']: r['violation'] for r in gt_raw}
    pred_frames = {v.frame_idx: v.violation for v in all_violations}

    all_frames = sorted(set(list(gt_frames.keys()) + list(pred_frames.keys())))
    y_true = [gt_frames.get(f,   'No violation') for f in all_frames]
    y_pred = [pred_frames.get(f, 'No violation') for f in all_frames]

    labels = VIOLATION_TYPES

    print('Classification Report:')
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix')
    plt.ylabel('Ground Truth')
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(str(REPORTS / 'confusion_matrix.png'), dpi=150)
    plt.show()

    # mAP proxy (per-class AP at 0.5 IoU threshold)
    per_class_f1 = f1_score(y_true, y_pred, labels=labels,
                            average=None, zero_division=0)
    map_score = float(np.mean(per_class_f1))
    print(f'\n  mAP@0.5 (proxy via F1): {map_score:.4f}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — DOWNLOAD ALL OUTPUTS
# ─────────────────────────────────────────────
import zipfile
from google.colab import files

zip_path = '/content/traffic_evidence.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Annotated frames
    for f in FRAMES_DIR.glob('*.jpg'):
        zf.write(f, f'frames/{f.name}')
    # Clips
    for f in CLIPS_DIR.glob('*.mp4'):
        zf.write(f, f'clips/{f.name}')
    # Reports
    for f in REPORTS.glob('*'):
        zf.write(f, f'reports/{f.name}')

print(f'✓ Packaged all outputs → {zip_path}')
files.download(zip_path)

## Next steps — improving accuracy

The current pipeline uses pretrained COCO weights. To reach production accuracy on traffic footage:

| Upgrade | How |
|---------|-----|
| Helmet detector | Fine-tune YOLOv8 on [IDD](https://idd.insaan.iiit.ac.in/) or [MHL dataset](https://github.com/onecycleLR/Helmet-Detection) |
| Seatbelt detector | Fine-tune on [SEATBELT dataset](https://universe.roboflow.com/seatbelt) |
| License plate OCR | Use [PaddleOCR](https://github.com/PaddlePaddle/PaddleOCR) + LP detector trained on Indian plates |
| Lane learning | Increase `LEARNING_FRAMES` to 3000+ for better cluster quality |
| Signal detection | Swap Layer 1 for a signal-specific YOLOv8 trained on traffic light crops |
| Speed | Switch master model to `yolov8n` + TensorRT export for 4–8× throughput |
